# 🚀 LLM (大規模言語モデル) 初心者向けガイド
**Python 開発経験者向け | 知識ゼロからのスタート**

---

## 📚 目次
1. [従来プログラミング vs AI の違い](#1-従来プログラミング-vs-ai-の違い)
2. [LLM の基本](#2-llm-の基本)
3. [ニューラルネットワークの仕組み](#3-ニューラルネットワークの仕組み)
4. [Transformer アーキテクチャ](#4-transformer-アーキテクチャ)
5. [学習フェーズ](#5-学習フェーズ)
6. [理解度チェック](#6-理解度チェック)

## 1. 従来プログラミング vs AI の違い

In [ ]:
# 従来プログラミング：ルールを人間が書く
def add(a, b):
    return a + b  # ルールが明示的

print("従来のアプローチ:", add(5, 3))

# AI 的アプローチ：データからパターンを学習
import numpy as np

# ダミーで「学習」を擬似的に表現
training_data = [(1,2,3), (4,5,9), (10,20,30)]  # (a, b, a+b)
weights = np.array([1.0, 1.0])  # 重みを学習

def ai_add(a, b):
    return float(np.dot([a, b], weights))

print("AI のアプローチ:", ai_add(5, 3))
print("\n→ 今は同じ結果でも、AI は「どんなルールか」を自分で見つけた点が違う")

## 2. LLM の基本

LLM の本質：「次の単語を予測する」を繰り返す

In [ ]:
# LLM の動作を単純なモデルで理解する
import random

# 超シンプルな「次の単語予測」モデル（概念用）
word_pairs = {
    "The quick": ["brown", "red", "blue"],
    "brown fox": ["jumps", "runs", "hides"],
    "Artificial intelligence": ["is", "will", "can"],
    "こんにちは、": ["元気ですか？", "世界！", "皆さん！"],
}

def simple_predict(context, temperature=0.5):
    candidates = word_pairs.get(context, ["..."])
    # temperature が高いほどランダム
    if temperature < 0.3:
        return candidates[0]  # 決定的
    return random.choice(candidates)  # ランダム

for ctx in ["The quick", "Artificial intelligence", "こんにちは、"]:
    pred = simple_predict(ctx)
    print(f"'{ctx}' → '{pred}'")

## 3. ニューラルネットワークの仕組み

In [ ]:
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

# 最小構成のニューラルネット（XOR 問題）
class TinyNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(2, 4), nn.ReLU(),
            nn.Linear(4, 1), nn.Sigmoid()
        )
    def forward(self, x): return self.net(x)

model = TinyNet()
optimizer = torch.optim.Adam(model.parameters(), lr=0.05)
loss_fn = nn.BCELoss()

# XOR データ
X = torch.tensor([[0,0],[0,1],[1,0],[1,1]], dtype=torch.float32)
y = torch.tensor([[0],[1],[1],[0]], dtype=torch.float32)

losses = []
for epoch in range(200):
    pred = model(X)
    loss = loss_fn(pred, y)
    optimizer.zero_grad(); loss.backward(); optimizer.step()
    losses.append(loss.item())

plt.figure(figsize=(7,3))
plt.plot(losses); plt.xlabel("Epoch"); plt.ylabel("Loss")
plt.title("ニューラルネットワークの学習曲線（XOR 問題）"); plt.tight_layout(); plt.show()

print("\n学習後の予測:")
with torch.no_grad():
    for xi, yi in zip(X, y):
        out = model(xi.unsqueeze(0)).item()
        print(f"  入力{xi.tolist()} → 予測: {out:.3f}  正解: {int(yi.item())}")

## 4. Transformer アーキテクチャ

### Tokenization → Attention → 次の単語予測

In [ ]:
from transformers import AutoTokenizer

# Tokenization を実際に動かす
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

text = "The quick brown fox jumps"
tokens = tokenizer.tokenize(text)
ids = tokenizer.convert_tokens_to_ids(tokens)

print("テキスト :", text)
print("トークン :", tokens)
print("ID       :", ids)
print()
print("CLS/SEP 付き:", tokenizer.encode(text, add_special_tokens=True))

In [ ]:
# Self-Attention の計算を手動で追体験
import torch
import torch.nn.functional as F

def scaled_dot_product_attention(Q, K, V):
    d_k = Q.size(-1)
    scores = torch.matmul(Q, K.transpose(-2, -1)) / (d_k ** 0.5)
    attn = F.softmax(scores, dim=-1)
    output = torch.matmul(attn, V)
    return output, attn

torch.manual_seed(0)
seq_len, d_k = 4, 8
Q = torch.randn(seq_len, d_k)
K = torch.randn(seq_len, d_k)
V = torch.randn(seq_len, d_k)

output, attn = scaled_dot_product_attention(Q, K, V)
print("Attention 重み（各トークンがどこに注目するか）:")
print(attn.round(decimals=2))
print("\n出力 shape:", output.shape)

## 5. 学習フェーズ

事前学習 → ファインチューニング → RLHF の 3 ステージ

In [ ]:
# 3段階の学習フェーズを損失関数の変化で可視化（シミュレーション）
import numpy as np
import matplotlib.pyplot as plt

def simulate_loss(n_steps, start, end, noise=0.02):
    curve = np.linspace(start, end, n_steps)
    curve += np.random.normal(0, noise, n_steps)
    return np.maximum(curve, 0)

np.random.seed(42)
pre   = simulate_loss(200, 4.5, 1.2, 0.15)
fine  = simulate_loss(100, 1.2, 0.4, 0.05)
rlhf  = simulate_loss( 80, 0.4, 0.15, 0.02)

all_loss = np.concatenate([pre, fine, rlhf])
xs = np.arange(len(all_loss))

fig, ax = plt.subplots(figsize=(9, 3))
ax.plot(xs[:200], pre,  label="Phase1: 事前学習",        color="#ff6b6b")
ax.plot(xs[200:300], fine, label="Phase2: ファインチューニング", color="#ffd93d")
ax.plot(xs[300:], rlhf, label="Phase3: RLHF",            color="#6bcb77")
ax.axvline(200, color="gray", ls="--", lw=0.8)
ax.axvline(300, color="gray", ls="--", lw=0.8)
ax.set_xlabel("ステップ"); ax.set_ylabel("損失")
ax.set_title("LLM の 3 段階学習フェーズ")
ax.legend(); plt.tight_layout(); plt.show()

## 6. 理解度チェック

- [ ] 従来プログラミングと AI の根本的な違いが説明できる  
- [ ] LLM が「次の単語を予測」していることが分かった  
- [ ] Attention の役割（文脈に応じた意味把握）が理解できた  
- [ ] 3 段階の学習フェーズが順番に言える  

---
✅ 完了 → **[段階2: プロジェクト全体像](02_project_overview_diagram.md)** へ